# 04 · Filtering & searching 🔍

This is **the core skill** of the whole project. A screener isn't much use if it
just dumps every row at you — the value is being able to ask questions like
*"which days did the price go up?"* or *"show me only the high-volume days"*.

That's **filtering**: keeping the rows that match a condition.

## What you'll learn
- **Boolean masks** — a column of `True`/`False`
- Keeping rows with `data[mask]`
- Combining conditions with `&` (and) / `|` (or) — **mind the brackets!**
- Sorting with `.sort_values(...)`
- Finding the best / worst day with `.idxmax()` / `.idxmin()`
- Filtering by date
- Filtering a **table of stocks** — the screener itself

> 🌐 This notebook downloads data, so you'll want internet. If you saved
> `aapl.csv` in notebook 03 you could load that instead.

## 0. Get some data to play with

We use the **same download form as notebook 03** — one ticker, flat column names —
so that `data["Close"]` is a single column and the filtering below behaves.

In [ ]:
import yfinance as yf

data = yf.download("AAPL", period="6mo", auto_adjust=True, multi_level_index=False, progress=False)
data.head()

## 1. Boolean masks — the big idea

Compare a whole column to a value and pandas checks **every row at once**, giving
you back a column of `True`/`False`. That column is called a **mask**.

Here: on which days did Apple close **higher than it opened** (an "up" day)?

In [ ]:
mask = data["Close"] > data["Open"]
mask.head()          # True where Close > Open, False otherwise

A neat trick: `True` counts as 1 and `False` as 0, so `.sum()` tells you **how
many** rows matched.

In [ ]:
print("Number of up days:", mask.sum())
print("Total days:       ", len(data))

## 2. Keep only the matching rows

Put the mask **inside square brackets** and pandas hands you back only the rows
where it was `True`. This is the move you'll use over and over.

In [ ]:
up_days = data[mask]
up_days.head()

You don't need to name the mask first — it's common to write the condition
straight inside the brackets:

In [ ]:
down_days = data[data["Close"] < data["Open"]]
print("Down days:", len(down_days))
down_days.head()

### 🏋️ Your turn

Keep only the **high-volume** days: rows where `Volume` is greater than the
**average** volume. (Hint: the average is `data["Volume"].mean()`.)

In [ ]:
# TODO: build a mask for Volume > average volume, then keep those rows.

**One way to do it:**

```python
busy = data[data["Volume"] > data["Volume"].mean()]
busy.head()
```

## 3. Combine conditions — `&` and `|`

Real questions often have **two** parts: *"an up day **and** a busy day"*.

- `&` means **and** (both must be true)
- `|` means **or** (either can be true)

🚨 **The golden rule:** wrap **each condition in its own brackets**. Miss them and
you'll get a confusing error. It looks like this:

```python
data[(condition A) & (condition B)]
```

In [ ]:
avg_volume = data["Volume"].mean()

busy_up_days = data[(data["Close"] > data["Open"]) & (data["Volume"] > avg_volume)]
print("Up AND busy:", len(busy_up_days))
busy_up_days.head()

With `|` you keep rows matching **either** condition.

In [ ]:
# Days that were EITHER a big up move OR a big down move (very busy either way)
big_moves = data[(data["Close"] > data["Open"] * 1.02) |
                 (data["Close"] < data["Open"] * 0.98)]
print("Days that moved more than 2%:", len(big_moves))
big_moves.head()

### 🏋️ Your turn

Find the days that were **both** an up day (`Close > Open`) **and** had a `Close`
above `200`. Remember the brackets around each condition!

In [ ]:
# TODO: combine (Close > Open) AND (Close > 200) with & and keep those rows.

**One way to do it:**

```python
result = data[(data["Close"] > data["Open"]) & (data["Close"] > 200)]
result.head()
```

(If your ticker never traded above 200 in this period you'll get an empty table —
that's a correct answer too! Try lowering the number.)

## 4. Sorting — `.sort_values`

Filtering keeps rows; **sorting** reorders them. Pass the column to sort by. Add
`ascending=False` to go **high → low** (great for "top" lists).

In [ ]:
# The 5 highest-volume days
data.sort_values("Volume", ascending=False).head()

### 🏋️ Your turn

Show the **5 days with the highest `Close`** price. (Sort by `"Close"`, biggest
first, then `.head()`.)

In [ ]:
# TODO: sort by Close descending and show the top 5.

**One way to do it:**

```python
data.sort_values("Close", ascending=False).head()
```

## 5. Best / worst day — `.idxmax` and `.idxmin`

`.max()` gives you the biggest **value**. But you often want to know **which day**
that happened on — the row's **label** (the date). That's what `.idxmax()` (index
of the max) and `.idxmin()` (index of the min) give you.

In [ ]:
best_day = data["Close"].idxmax()      # the DATE of the highest close
worst_day = data["Close"].idxmin()     # the DATE of the lowest close

print("Highest close was on:", best_day)
print("Lowest close was on: ", worst_day)

Feed that date back into `.loc` to see the whole row for that day.

In [ ]:
data.loc[best_day]         # the full row for the best day

### 🏋️ Your turn

Find the date of the **highest-volume** day, then use `.loc` to print that whole
row.

In [ ]:
# TODO: use .idxmax() on the Volume column, then .loc that date.

**One way to do it:**

```python
busiest = data["Volume"].idxmax()
print("Busiest day:", busiest)
data.loc[busiest]
```

## 6. Filter by date

Because the **date is the index**, you can filter by it with `.loc` and a date
written as text (`"YYYY-MM-DD"`). This keeps every row **on or after** that date.

In [ ]:
recent = data.loc["2026-05-01":]      # everything from 1 May 2026 onwards
print(recent.shape)
recent.head()

> 💡 Pick a date that's actually inside your downloaded range. If you get an empty
> result, choose an earlier date. You can see the range with
> `print(data.index.min(), "to", data.index.max())`.

In [ ]:
print(data.index.min(), "to", data.index.max())

## 7. Filtering a table of stocks (a screener!) 🔎

Everything above filtered **one company's daily rows**. Now the payoff: the *exact
same moves* work on a table where **each row is a whole company**. That table is a
**screener** — look at many stocks at once and filter down to the interesting ones.

First, let's build a small `universe` to play with. In the real app it comes from a
multi-ticker download (notebook 03); here we type in a handful of made-up figures
so the results below are predictable and this section needs no internet. It has the
**same shape** as the `summary` from notebook 03: indexed by ticker, with `Name`,
`Price`, `Change %` and `Avg Volume` columns.

In [ ]:
import pandas as pd

universe = pd.DataFrame(
    {
        "Name": ["Apple", "Microsoft", "Tesla", "Nvidia", "Coca-Cola"],
        "Price": [210.50, 430.20, 240.80, 125.60, 68.30],
        "Change %": [8.30, 12.10, -15.40, 45.20, -2.10],
        "Avg Volume": [55_000_000, 22_000_000, 98_000_000, 210_000_000, 14_000_000],
    },
    index=["AAPL", "MSFT", "TSLA", "NVDA", "KO"],
)
universe

### Only the risers

The mask move is identical — just point it at the stock table. Keep the companies
whose `Change %` is positive:

In [ ]:
universe[universe["Change %"] > 0]

### Rank them best to worst

Sorting works the same way here. Order the whole table by `Change %`, biggest
first:

In [ ]:
universe.sort_values("Change %", ascending=False)

### Combine conditions on the stock table

Same `&`, and the **same golden rule** — each condition in its own brackets. Here:
risers **and** heavily traded (a big average volume):

In [ ]:
universe[(universe["Change %"] > 0) & (universe["Avg Volume"] > 50_000_000)]

### Search by name

A screener usually has a search box. `.str.contains(...)` checks each name for a
piece of text and hands back a mask — so it slots straight into the brackets.
(It's case-sensitive, so `"Co"` finds Coca-Cola.)

In [ ]:
universe[universe["Name"].str.contains("Co")]

> 🧭 **This row-filtering IS what the screener app does.** A slider, a checkbox, a
> search box — each one just builds a mask like these and hands you
> `universe[mask]`. You've already written the hard part.

### 🏋️ Your turn (1)

Find the **3 highest-volume** stocks in the `universe`. (Sort by `"Avg Volume"`,
biggest first, then take the top 3.)

In [ ]:
# TODO: sort universe by Avg Volume, descending, and keep the top 3 rows.

**One way to do it:**

```python
universe.sort_values("Avg Volume", ascending=False).head(3)
```

### 🏋️ Your turn (2)

Keep only the stocks that **fell** over the period (a negative `Change %`).

In [ ]:
# TODO: keep the rows where Change % is below 0.

**One way to do it:**

```python
universe[universe["Change %"] < 0]
```

## 🎉 This is the big one — you've got it!

Boolean masks, `data[mask]`, `&`/`|` with brackets, sorting, `idxmax`/`idxmin`,
date slices — and the same moves applied to a **table of stocks**. Every search or
filter in the screener is one of these patterns: build a mask, show
`universe[mask]`.

**Next up:** [`05_charts_and_streamlit_prep.ipynb`](05_charts_and_streamlit_prep.ipynb)
— turn a stock into a picture, and see how the whole screener becomes the app.

📖 Companion guide:
[`../guides/04-filtering-searching.md`](../guides/04-filtering-searching.md)